# OpenWebText Multi-Xi SPLM (d=256, L=12)

## Motivation

The TinyStories SPLM checkpoints (d=256, L=8, ~16.5M params) lack sufficient
semantic depth for Experiment B (hallucination / inconsistency detection):
shallow attractor basins, no world knowledge, and the resulting
$\Delta E_{\text{anomaly}}$ signal has insufficient dynamic range.

This notebook trains a **d=256, L=12** Multi-Xi SPLM on **OpenWebText** — the
same web corpus GPT-2 was trained on — to produce a model with:
- Factual content it can be inconsistent about
- Deeper integration (12 layers) for a richer energy landscape
- ~25-30M params — still small enough for A100/H100 training in <24h

## Config summary

| Parameter | Value |
|-----------|-------|
| Dataset | `Skylion007/openwebtext` (streamed, ~200M tokens) |
| Tokenizer | GPT-2 BPE (vocab=50257) |
| `d` | 256 |
| `L` | **12** (vs 8 for TinyStories) |
| `v_hidden` | 1024 |
| `v_depth` | 3 |
| `fixed_gamma` | 0.30 |
| `xi_channels` | 4 |
| Optimizer | AdamW (betas=0.9/0.95, wd=0.01) |
| LR | 5e-4 cosine, 2000 warmup |
| Total steps | 50,000 |
| Batch size | auto (8 for A100/H100, 4 for smaller GPUs) |
| Block size | 512 |
| Checkpoints | 10k, 25k, 50k steps |

Estimated wall time: **8-14 hours on A100**.

In [ ]:
# ── Cell 1: Environment + Drive mount ─────────────────────────────
import subprocess, sys, os, gc, math, json, time
from pathlib import Path
from dataclasses import asdict, fields as dc_fields

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow datasets')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    GPU_NAME = torch.cuda.get_device_name()
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_splm_openwebtext')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_splm_openwebtext'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = DRIVE_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive root   : {DRIVE_ROOT}')
print(f'Checkpoints  : {CKPT_DIR}')
print(f'Results      : {RESULTS_DIR}')

In [ ]:
# ── Cell 2: OpenWebText data loading (streaming, ~200M tokens) ────
# Stream from HuggingFace to avoid downloading the full ~50GB corpus.
# Tokenize in chunks, cache to .npy for fast reload on restarts.
# ┌─────────────────────────────────────────────────────────────────┐
# │  CONFIG: adjust MAX_TRAIN_TOKENS to trade training data vs time │
# └─────────────────────────────────────────────────────────────────┘
MAX_TRAIN_TOKENS = 200_000_000   # ~200M tokens (~1% of full OpenWebText)
VAL_TOKENS       = 2_000_000    # ~2M tokens for validation
CHUNK_SIZE       = 50_000       # documents per tokenization chunk

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens)...')

    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)

    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()

    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break

    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts

    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')

    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids

    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

gc.collect()
print(f'\nReady: train={len(train_ids):,}  val={len(val_ids):,}')

In [ ]:
# ── Cell 3: Compute logfreq surprisal for OpenWebText ─────────────
# Same add-one Laplace smoothing as the TinyStories version.

VOCAB_SIZE = 50257
LOGFREQ_PATH = str(DATA_DIR / 'logfreq_surprisal_openwebtext.npy')

if os.path.exists(LOGFREQ_PATH):
    print(f'Logfreq file exists: {LOGFREQ_PATH}')
else:
    print('Computing unigram surprisal from OpenWebText training tokens...')
    counts = np.bincount(train_ids, minlength=VOCAB_SIZE).astype(np.int64)
    N = len(train_ids)
    nz = int((counts > 0).sum())
    print(f'  Corpus tokens: {N:,}  Unique types: {nz:,} / {VOCAB_SIZE:,} '
          f'({100 * nz / VOCAB_SIZE:.1f}%)')

    p = (counts + 1.0) / (N + VOCAB_SIZE)
    surprisal = -np.log(p).astype(np.float32)
    print(f'  Surprisal: min={surprisal.min():.3f}  max={surprisal.max():.3f}  '
          f'mean={surprisal.mean():.3f}  median={np.median(surprisal):.3f}')

    np.save(LOGFREQ_PATH, surprisal)
    print(f'  Saved: {LOGFREQ_PATH}')
    del counts, p, surprisal

print('Logfreq surprisal ready.')

In [ ]:
# ── Cell 4: Model config (d=256, L=12) + instantiation ────────────
from model_multixi import (
    ScalarPotentialLMSARFMassLNMultiXi,
    SPLMSARFMassLNMultiXiConfig,
)

model_cfg = SPLMSARFMassLNMultiXiConfig(
    d=256,
    max_len=1024,
    v_hidden=1024,
    v_depth=3,
    L=12,
    init_m=1.0,
    init_gamma=1.0,
    vocab_size=VOCAB_SIZE,
    mass_mode='logfreq',
    logfreq_init_alpha=0.1,
    logfreq_path=LOGFREQ_PATH,
    ln_after_step=True,
    fixed_gamma=0.30,
    xi_channels=4,
    xi_alpha_inits=[0.0, 0.5, 0.9, 0.99],
    xi_learnable=True,
    causal_force=True,
    xi_alpha_init_mode='explicit',
    xi_tau_max=100.0,
)

model = ScalarPotentialLMSARFMassLNMultiXi(model_cfg).to(DEVICE)
n_params = model.num_params()
alpha_init_str = ','.join(f'{a:.3f}' for a in model.xi_alpha_values())

print(f'Model: ScalarPotentialLMSARFMassLNMultiXi')
print(f'  params: {n_params:,}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  v_hidden={model_cfg.v_hidden}  v_depth={model_cfg.v_depth}')
print(f'  xi_channels={model_cfg.xi_channels}  alpha_init=[{alpha_init_str}]  learnable={model_cfg.xi_learnable}')
print(f'  fixed_gamma={model_cfg.fixed_gamma}  mass_mode={model_cfg.mass_mode}')
print(f'  causal_force={model_cfg.causal_force}  ln_after_step={model_cfg.ln_after_step}')

In [ ]:
# ── Cell 5: Training loop (50k steps, checkpoints at 10k/25k/50k) ─
from data_module import get_batch

# ── auto-detect batch size based on available VRAM ──
if DEVICE == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 8 if vram >= 30 else 4
    print(f'Auto batch size: {BATCH_SIZE} (VRAM={vram:.1f} GB)')
else:
    BATCH_SIZE = 4
    print(f'Batch size: {BATCH_SIZE} (non-CUDA device)')

BLOCK_SIZE    = 512
TOTAL_STEPS   = 50_000
LR            = 5e-4
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 2000
GRAD_CLIP     = 1.0
EVAL_INTERVAL = 1000
EVAL_ITERS    = 40
LOG_INTERVAL  = 100
CKPT_STEPS    = {10_000, 25_000, 50_000}
SEED          = 0

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

def lr_schedule(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
        },
        'step': step_num,
        'final_val_loss': val_loss_val,
        'final_val_ppl': math.exp(val_loss_val),
        'final_gamma': model.gamma.item(),
        'final_xi_alphas': model.xi_alpha_values(),
        'fixed_gamma': model_cfg.fixed_gamma,
        'logfreq_path': LOGFREQ_PATH,
        'variant': 'sarf_mass_ln_multixi',
        'experiment': 'openwebtext_splm_d256_L12',
        'corpus': 'openwebtext',
        'seed': SEED,
    }
    fname = f'splm_multixi_owt_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    torch.save(ckpt, path)
    print(f'  Checkpoint saved: {path}')
    return path

# ── resume support: check if a partially-trained checkpoint exists ──
resume_step = 0
resume_ckpt = None
for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'splm_multixi_owt_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

optim = torch.optim.AdamW(
    model.parameters(), lr=LR,
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
)

if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'])
    print(f'  Model loaded. Starting from step {resume_step}.')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── training log ──
log_path = RESULTS_DIR / 'training_log.jsonl'
log_f = log_path.open('a')
loss_history = []

t0 = time.time()
model.train()
running = 0.0
n_run = 0

print(f'\n{"="*60}')
print(f'Training: steps {resume_step+1} -> {TOTAL_STEPS}')
print(f'  batch={BATCH_SIZE}  block={BLOCK_SIZE}  lr={LR}  warmup={WARMUP_STEPS}')
print(f'  checkpoints at steps: {sorted(CKPT_STEPS)}')
print(f'{"="*60}\n')

for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
    x = torch.from_numpy(xb).to(DEVICE)
    y = torch.from_numpy(yb).to(DEVICE)

    _, loss = model(x, y)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optim.step()

    running += loss.item()
    n_run += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg = running / n_run
        running, n_run = 0.0, 0
        elapsed = time.time() - t0
        gamma_val = model.gamma.item()
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        print(
            f'step {step+1:6d}/{TOTAL_STEPS}  '
            f'train={avg:.4f}  lr={lr_now:.2e}  grad={float(grad_norm):.2f}  '
            f'gamma={gamma_val:.3f}  alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s'
        )
        log_f.write(json.dumps({
            'step': step + 1, 'train_loss': avg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'gamma': gamma_val, 'xi_alphas': alphas,
            'elapsed_sec': elapsed,
        }) + '\n')
        log_f.flush()

    if (step + 1) % EVAL_INTERVAL == 0:
        vl = evaluate()
        vppl = math.exp(vl)
        elapsed = time.time() - t0
        print(f'  >>> EVAL step {step+1}  val_loss={vl:.4f}  val_ppl={vppl:.2f}  ({elapsed:.0f}s)')
        loss_history.append((step + 1, running / max(n_run, 1) if n_run > 0 else avg, vl))
        log_f.write(json.dumps({
            'step': step + 1, 'val_loss': vl, 'val_ppl': vppl,
        }) + '\n')
        log_f.flush()

    if (step + 1) in CKPT_STEPS:
        vl = evaluate()
        save_checkpoint(step + 1, vl)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

log_f.close()
print(f'\nTraining complete ({time.time() - t0:.0f}s total).')

In [ ]:
# ── Cell 6: Final evaluation + summary ────────────────────────────
final_val = evaluate()
final_ppl = math.exp(final_val)
final_gamma = model.gamma.item()
final_alphas = model.xi_alpha_values()
total_elapsed = time.time() - t0

print(f'\n{"="*60}')
print(f'FINAL  val_loss={final_val:.4f}  val_ppl={final_ppl:.2f}')
print(f'  gamma={final_gamma:.4f}')
print(f'  alpha_final={final_alphas}')
print(f'  elapsed={total_elapsed:.0f}s ({total_elapsed/3600:.2f}h)')
print(f'{"="*60}')

# Save final checkpoint as "best"
save_checkpoint(TOTAL_STEPS, final_val, tag_suffix='_final')

# ── loss curve plot ──
if loss_history:
    steps_v, train_vs, val_vs = zip(*loss_history)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps_v, val_vs, 'o-', color='darkorange', label='val loss')
    ax1.set_xlabel('step')
    ax1.set_ylabel('loss (nats)')
    ax1.set_title('Validation loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(steps_v, [math.exp(v) for v in val_vs], 'o-', color='royalblue', label='val ppl')
    ax2.set_xlabel('step')
    ax2.set_ylabel('perplexity')
    ax2.set_title('Validation perplexity')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    fig.suptitle(
        f'SPLM Multi-Xi OpenWebText  d={model_cfg.d}  L={model_cfg.L}  '
        f'gamma={final_gamma:.3f}  params={n_params:,}',
        fontsize=11,
    )
    fig.tight_layout()
    fig_path = RESULTS_DIR / 'training_curve.png'
    fig.savefig(fig_path, dpi=150)
    print(f'Loss curve saved: {fig_path}')
    plt.show()

# ── summary markdown ──
summary_path = RESULTS_DIR / 'training_summary.md'
with summary_path.open('w') as f:
    f.write('# Training summary — SPLM Multi-Xi OpenWebText\n\n')
    f.write('- experiment: OpenWebText SPLM d=256 L=12\n')
    f.write('- model: ScalarPotentialLMSARFMassLNMultiXi\n')
    f.write(f'- corpus: OpenWebText (~{MAX_TRAIN_TOKENS//1_000_000}M train tokens)\n')
    f.write(f'- params: {n_params:,}\n')
    f.write(f'- d={model_cfg.d}  L={model_cfg.L}  v_hidden={model_cfg.v_hidden}\n')
    f.write(f'- xi_channels={model_cfg.xi_channels}  alpha_final={final_alphas}\n')
    f.write(f'- fixed_gamma: {model_cfg.fixed_gamma}\n')
    f.write(f'- batch_size={BATCH_SIZE}  block_size={BLOCK_SIZE}  steps={TOTAL_STEPS}\n')
    f.write(f'- seed: {SEED}\n')
    f.write(f'- elapsed: {total_elapsed:.0f}s ({total_elapsed/3600:.2f}h)\n\n')
    f.write(f'Final val loss: {final_val:.6f} (ppl {final_ppl:.2f})\n')
    f.write(f'Final gamma: {final_gamma:.4f}\n')
    f.write(f'Final alpha: {final_alphas}\n')
print(f'Summary: {summary_path}')

In [ ]:
# ── Cell 7: (Optional) Push checkpoint to HuggingFace Hub ─────────
# Set PUSH_TO_HF = True and fill in your HF token to upload.
PUSH_TO_HF = False
HF_REPO     = 'dimitarpg13/semsimula-splm-multixi-owt'
HF_TOKEN    = ''   # paste your HF write token here, or set HF_TOKEN env var

if PUSH_TO_HF:
    from huggingface_hub import HfApi, create_repo
    token = HF_TOKEN or os.environ.get('HF_TOKEN', '')
    if not token:
        print('No HF token — skipping push. Set HF_TOKEN env var or paste above.')
    else:
        api = HfApi(token=token)
        try:
            create_repo(HF_REPO, repo_type='model', exist_ok=True, token=token)
        except Exception as e:
            print(f'  (repo may already exist: {e})')

        final_ckpt = CKPT_DIR / f'splm_multixi_owt_step{TOTAL_STEPS}_final.pt'
        if final_ckpt.exists():
            print(f'Uploading {final_ckpt} to {HF_REPO}...')
            api.upload_file(
                path_or_fileobj=str(final_ckpt),
                path_in_repo=final_ckpt.name,
                repo_id=HF_REPO,
                token=token,
            )
            print('Upload complete.')
        else:
            print(f'Final checkpoint not found at {final_ckpt}')

        logfreq_file = DATA_DIR / 'logfreq_surprisal_openwebtext.npy'
        if logfreq_file.exists():
            print(f'Uploading logfreq file...')
            api.upload_file(
                path_or_fileobj=str(logfreq_file),
                path_in_repo=logfreq_file.name,
                repo_id=HF_REPO,
                token=token,
            )
            print('Logfreq upload complete.')
else:
    print('HF push disabled. Set PUSH_TO_HF = True to upload.')